# Create a service principal, deploy a Declarative Automation Bundle as the principal, and confirm the audit log attributes the run to the principal

The SOC 2 auditor wants evidence that the prod deploys come from a service identity, not from a human's PAT. The team has been using the lead engineer's PAT for the past quarter because the platform did not have time to wire M2M auth, and the auditor is unimpressed. You have one session to spin up the service principal, deploy the Lab 8 bundle as the principal, prove the audit log surfaces the principal as the actor, and write a short runbook the rest of the team can use to migrate their existing CI/CD pipelines.

The hands-on work:

- Create a service principal `labrun-cicd-deployer` at the account level via SCIM.
- Generate an OAuth M2M client_id + client_secret for the principal (account console; the secret cannot be retrieved later, so capture it once via getpass).
- Assign the SP to the trial workspace (workspace permission-assignment).
- Configure a local Databricks CLI profile `labrun-sp` with the SP credentials.
- Re-use the Lab 8 bundle directory and deploy the prod target as the SP.
- Trigger one run via `databricks jobs run-now --job-id <id> --profile labrun-sp` and wait for SUCCESS.
- Query `system.access.audit` and confirm the actor on both `bundle.deploy` and `jobs.runNow` events is the SP's identifier, NOT your SSO email.

**Time.** Plan for about 75 minutes hands-on. The audit log lag (1 to 5 minutes after the event) adds idle time at the end. Budget up to 130 minutes for the session window.

**Cost.** $0.10 to $0.50 per session on your cloud account. This is the LAST Premium-trial lab in the cert track; tear down the trial workspace before day 14 of the trial or it converts to paid.

This lab maps to Databricks DE Associate Domain 5 (CI/CD, ~10 percent provisional) and Domain 7 (Governance and Security, ~11 percent provisional).

**Rename callout.** If your other prep material says "Databricks Asset Bundles" or "DABs," the new product name on the current exam is **Declarative Automation Bundles**. The CLI subcommand is still `databricks bundle ...` and the file is still `databricks.yml`. If your prep material says "audit logs are delivered to a customer S3 bucket only," that is outdated. The current exam ALSO covers `system.access.audit` (the Unity Catalog system table holding the audit log inline for SQL query).

**Local prerequisites.**

- Databricks CLI v0.220 or later installed locally and on your PATH.
- The Lab 08 bundle directory `labrun-bundle/` on your local machine (or you can re-init it via `databricks bundle init` with the default-python template; the lab uses any bundle with at least one job).
- Workspace URL, workspace PAT, account host, account ID, account-level admin PAT (same five getpass prompts as Lab 11).
- The SP's OAuth client_id and client_secret (you will generate these in Task 1 and paste them when needed).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json as _json
import sys
import time

import requests

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "service-principal-cicd-and-audit-trail"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"

SP_DISPLAY_NAME = "labrun-cicd-deployer"
CLI_PROFILE_NAME = "labrun-sp"
JOB_NAME_HINT = "labrun-bundle-job"  # The job's name in Lab 8's prod target.

# Filled by the student.
workspace_id = None
sp_internal_id = None        # SCIM internal id
sp_application_id = None     # OAuth client_id; what the SP authenticates with
prod_job_id = None           # The Lab 8 prod-target job after the SP re-deploys
run_id = None                # The triggered run's run_id

In [ ]:
# NBVAL_SKIP
# Register the session, validate workspace + account credentials, capture
# workspace_id from the workspace URL, expose run_sql() and SCIM helpers.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks workspace PAT: ").strip()
account_host = getpass.getpass("Databricks account console host (e.g. https://accounts.cloud.databricks.com): ").strip()
account_id = getpass.getpass("Databricks account ID (UUID): ").strip()
account_token = getpass.getpass("Databricks account admin PAT: ").strip()

for name, val in (
    ("workspace URL", databricks_host),
    ("workspace PAT", databricks_token),
    ("account host", account_host),
    ("account id", account_id),
    ("account PAT", account_token),
):
    if not val:
        print(f"{name} is required. Re-run this cell.")
        raise SystemExit(1)

if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"
databricks_host = databricks_host.rstrip("/")
if not account_host.startswith("https://"):
    account_host = f"https://{account_host}"
account_host = account_host.rstrip("/")

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Workspace credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Workspace user: {CALLER_USER}")

# Resolve the workspace_id via the account workspaces API.
ws_resp = requests.get(
    f"{account_host}/api/2.0/accounts/{account_id}/workspaces",
    headers={"Authorization": f"Bearer {account_token}"},
    timeout=30,
)
if ws_resp.status_code != 200:
    print(f"Could not list account workspaces: {ws_resp.status_code} {ws_resp.text[:200]}")
    raise SystemExit(1)
workspaces = ws_resp.json() if isinstance(ws_resp.json(), list) else ws_resp.json().get("workspaces", [])
hostname_for_match = databricks_host.replace("https://", "").rstrip("/")
for ws in workspaces:
    if (ws.get("deployment_name") and ws["deployment_name"] in hostname_for_match) or (
        ws.get("workspace_url", "").replace("https://", "").rstrip("/") == hostname_for_match
    ):
        workspace_id = ws.get("workspace_id")
        break
if workspace_id is None and workspaces:
    workspace_id = workspaces[0].get("workspace_id")
print(f"Workspace id: {workspace_id}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Create one and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
    "account_host": account_host,
    "account_id": account_id,
    "account_token": account_token,
    "workspace_id": workspace_id,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180, ignore_errors=False):
    wid = warehouse_id or WAREHOUSE_ID
    try:
        resp = w.statement_execution.execute_statement(
            warehouse_id=wid, statement=statement, wait_timeout="50s",
        )
        statement_id = resp.statement_id
        deadline = time.time() + wait_seconds
        while True:
            state = resp.status.state if resp.status else None
            if state == StatementState.SUCCEEDED:
                break
            if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
                err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
                if ignore_errors:
                    return {"columns": [], "rows": [], "statement_id": statement_id, "error": err}
                raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement[:200]}")
            if time.time() > deadline:
                raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
            time.sleep(2)
            resp = w.statement_execution.get_statement(statement_id)
        columns = []
        if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
            columns = [c.name for c in resp.manifest.schema.columns]
        rows = []
        if resp.result and resp.result.data_array:
            rows = list(resp.result.data_array)
        return {"columns": columns, "rows": rows, "statement_id": statement_id, "error": None}
    except Exception as exc:
        if ignore_errors:
            return {"columns": [], "rows": [], "statement_id": None, "error": str(exc)}
        raise


def acct_get(path, params=None):
    url = f"{account_host}/api/2.0/accounts/{account_id}/{path}"
    return requests.get(url, headers={"Authorization": f"Bearer {account_token}"}, params=params or {}, timeout=30)


def acct_post(path, body):
    url = f"{account_host}/api/2.0/accounts/{account_id}/{path}"
    return requests.post(
        url, headers={"Authorization": f"Bearer {account_token}", "Content-Type": "application/json"},
        data=_json.dumps(body), timeout=30,
    )


def acct_put(path, body):
    url = f"{account_host}/api/2.0/accounts/{account_id}/{path}"
    return requests.put(
        url, headers={"Authorization": f"Bearer {account_token}", "Content-Type": "application/json"},
        data=_json.dumps(body), timeout=30,
    )


def acct_delete(path):
    url = f"{account_host}/api/2.0/accounts/{account_id}/{path}"
    return requests.delete(url, headers={"Authorization": f"Bearer {account_token}"}, timeout=30)


def scim_get(path, params=None):
    return acct_get(f"scim/v2/{path}", params=params)


def scim_post(path, body):
    return requests.post(
        f"{account_host}/api/2.0/accounts/{account_id}/scim/v2/{path}",
        headers={
            "Authorization": f"Bearer {account_token}",
            "Content-Type": "application/scim+json",
        },
        data=_json.dumps(body), timeout=30,
    )


def scim_delete(path):
    return acct_delete(f"scim/v2/{path}")


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest + adapter + atexit + orphan scan. Critical order:
# job (consumes deploy quota) -> workspace bundle path -> OAuth secret ->
# workspace assignment -> SP itself. CLI profile is a local reminder.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="lakeflow_job_by_name",
        id=JOB_NAME_HINT,
        region="databricks",
        cli_delete_command="cd labrun-bundle && databricks bundle destroy --target prod --profile labrun-sp --auto-approve",
    ),
    CleanupResource(
        type="workspace_path",
        id=f"/Workspace/Users/{SP_DISPLAY_NAME}",
        region="databricks",
        cli_delete_command=f"databricks workspace delete --recursive /Workspace/Users/{SP_DISPLAY_NAME}",
    ),
    CleanupResource(
        type="service_principal_oauth_secret",
        id=f"sp:{SP_DISPLAY_NAME}",
        region="databricks",
        cli_delete_command=f"databricks account service-principal-secrets delete <SP_APPLICATION_ID> <SECRET_ID>",
    ),
    CleanupResource(
        type="service_principal_workspace_assignment",
        id=f"workspace_assignment:{SP_DISPLAY_NAME}",
        region="databricks",
        cli_delete_command=f"databricks workspace-assignment delete <WORKSPACE_ID> <SP_APPLICATION_ID>",
    ),
    CleanupResource(
        type="service_principal",
        id=SP_DISPLAY_NAME,
        region="databricks",
        cli_delete_command=f"databricks account service-principals delete <SP_APPLICATION_ID>",
    ),
    CleanupResource(
        type="databricks_cli_profile_reminder",
        id=CLI_PROFILE_NAME,
        region="databricks",
        cli_delete_command=(
            "echo 'Manually remove the [labrun-sp] block from "
            "~/.databrickscfg; the cleanup cell does not edit local files.'"
        ),
    ),
]


def _find_sp_by_display_name(name):
    resp = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{name}"'})
    if resp.status_code != 200:
        return None
    items = resp.json().get("Resources") or []
    return items[0] if items else None


def _find_job_by_name_contains(needle):
    found = []
    try:
        for job in w.jobs.list():
            n = (job.settings.name if job.settings else "") or ""
            if needle in n:
                found.append(job)
    except Exception:
        pass
    return found


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "lakeflow_job_by_name":
            for job in _find_job_by_name_contains(rid):
                try:
                    runs = w.jobs.list_runs(job_id=job.job_id, active_only=True)
                    for run in (runs.runs or []):
                        try:
                            w.jobs.cancel_run(run.run_id)
                        except Exception:
                            pass
                except Exception:
                    pass
                try:
                    w.jobs.delete(job.job_id)
                except Exception:
                    pass
        elif rtype == "workspace_path":
            try:
                w.workspace.delete(rid, recursive=True)
            except Exception:
                pass
        elif rtype == "service_principal_oauth_secret":
            sp = _find_sp_by_display_name(SP_DISPLAY_NAME)
            if sp:
                app_id = sp.get("applicationId")
                if app_id:
                    secrets_resp = acct_get(f"servicePrincipals/{app_id}/credentials/secrets")
                    if secrets_resp.status_code == 200:
                        for s in (secrets_resp.json().get("secrets") or []):
                            secret_id = s.get("id")
                            if secret_id:
                                acct_delete(f"servicePrincipals/{app_id}/credentials/secrets/{secret_id}")
        elif rtype == "service_principal_workspace_assignment":
            sp = _find_sp_by_display_name(SP_DISPLAY_NAME)
            if sp and workspace_id:
                app_id = sp.get("applicationId")
                if app_id:
                    acct_delete(f"workspaces/{workspace_id}/permissionassignments/principals/{app_id}")
        elif rtype == "service_principal":
            sp = _find_sp_by_display_name(SP_DISPLAY_NAME)
            if sp:
                sp_id = sp.get("id")
                if sp_id:
                    scim_delete(f"ServicePrincipals/{sp_id}")
        elif rtype == "databricks_cli_profile_reminder":
            # No automated delete; just print reminder.
            pass
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "lakeflow_job_by_name":
            return len(_find_job_by_name_contains(rid)) > 0
        if rtype == "workspace_path":
            try:
                w.workspace.get_status(rid)
                return True
            except Exception:
                return False
        if rtype == "service_principal":
            return _find_sp_by_display_name(SP_DISPLAY_NAME) is not None
        if rtype == "service_principal_oauth_secret":
            sp = _find_sp_by_display_name(SP_DISPLAY_NAME)
            if not sp:
                return False
            app_id = sp.get("applicationId")
            if not app_id:
                return False
            secrets_resp = acct_get(f"servicePrincipals/{app_id}/credentials/secrets")
            return secrets_resp.status_code == 200 and bool((secrets_resp.json().get("secrets") or []))
        if rtype == "service_principal_workspace_assignment":
            sp = _find_sp_by_display_name(SP_DISPLAY_NAME)
            if not sp or not workspace_id:
                return False
            app_id = sp.get("applicationId")
            if not app_id:
                return False
            resp = acct_get(f"workspaces/{workspace_id}/permissionassignments")
            if resp.status_code != 200:
                return False
            for pa in (resp.json().get("permission_assignments") or []):
                principal = pa.get("principal") or {}
                if str(principal.get("service_principal_id") or principal.get("application_id")) == str(app_id):
                    return True
            return False
        if rtype == "databricks_cli_profile_reminder":
            return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        try:
            resp = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'})
            if resp.status_code == 200:
                for sp in (resp.json().get("Resources") or []):
                    arns.append(f"sp:{sp.get('displayName')}")
        except Exception:
            pass
        for job in _find_job_by_name_contains(JOB_NAME_HINT):
            tags = (job.settings.tags if job.settings else None) or {}
            if tags.get(LAB_TAG_KEY) == lab_slug:
                arns.append(f"job:{job.settings.name}")
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the service principal, generate the OAuth secret, assign the SP to the workspace

Three steps, all against the account API.

**Step A.** Create the service principal at the account level via SCIM. Capture the SCIM `id` (internal) and the `applicationId` (which is the OAuth `client_id`).

**Step B.** Generate an OAuth M2M secret for the principal via the account-side endpoint `POST /api/2.0/accounts/{account_id}/servicePrincipals/{application_id}/credentials/secrets`. The response includes a `secret` field that is returned ONCE and never again. Save it via `getpass()` (or paste it into a CLI profile in Task 2); never echo it to a cell output.

**Step C.** Assign the SP to the trial workspace. The endpoint is `PUT /api/2.0/accounts/{account_id}/workspaces/{workspace_id}/permissionassignments/principals/{principal_id}` with a body that grants `USER` (or `ADMIN`) permission. The lab uses `USER` plus a separate `CAN MANAGE` grant on the bundle's target job in Task 2.

Note: The SCIM API can create the SP but the SECRET CREATE is a separate non-SCIM endpoint. Both are account-scoped.

In [ ]:
# NBVAL_SKIP
# Task 1: Create SP, generate OAuth M2M secret, assign to workspace.

global sp_internal_id, sp_application_id

# YOUR CODE: Create the SP via scim_post.
# Example:
#   sp_resp = scim_post("ServicePrincipals", {
#       "schemas": ["urn:ietf:params:scim:schemas:core:2.0:ServicePrincipal"],
#       "displayName": SP_DISPLAY_NAME,
#       "active": True,
#   })
#   if sp_resp.status_code == 201:
#       sp_payload = sp_resp.json()
#   else:
#       sp_payload = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'}).json()["Resources"][0]
#   sp_internal_id = sp_payload["id"]
#   sp_application_id = sp_payload.get("applicationId")

if not sp_application_id:
    print("sp_application_id is None. Create the SP and capture applicationId.")
    raise SystemExit(1)

# YOUR CODE: Create the OAuth M2M secret via the account endpoint.
# Print a one-time getpass-style line so the secret is captured safely:
# Example:
#   secret_resp = acct_post(
#       f"servicePrincipals/{sp_application_id}/credentials/secrets", {}
#   )
#   if secret_resp.status_code in (200, 201):
#       data = secret_resp.json()
#       sp_oauth_secret = data.get("secret")
#       sp_oauth_secret_id = data.get("id")
#       # DO NOT print sp_oauth_secret here. Write it ONLY to your CLI
#       # profile in Task 2.

# YOUR CODE: Assign the SP to the workspace via PUT.
# Example:
#   acct_put(
#       f"workspaces/{workspace_id}/permissionassignments/principals/{sp_application_id}",
#       {"permissions": ["USER"]},
#   )

print(f"Service principal SCIM id:  {sp_internal_id}")
print(f"Service principal client_id (applicationId): {sp_application_id}")
print(f"Workspace assignment target: workspace {workspace_id}")
print()
print("CAPTURE THE OAUTH SECRET NOW. It is returned once and never again.")
print("Paste it into your local CLI profile (~/.databrickscfg) under [labrun-sp]")
print("with auth_type = oauth-m2m. See Task 2 for the canonical block.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: SP exists at account level, has at least one OAuth secret,
# and is assigned to the trial workspace.


def checkpoint_1(session):
    try:
        if not sp_application_id:
            return CheckpointResult(
                status="fail",
                error_reason="sp_application_id is None. Create the SP and capture applicationId.",
            )
        sp_resp = scim_get(
            "ServicePrincipals",
            params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'},
        )
        if sp_resp.status_code != 200 or not (sp_resp.json().get("Resources") or []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Service principal {SP_DISPLAY_NAME!r} not found via account SCIM. "
                    f"Confirm the account token has account admin scope."
                ),
            )

        secrets_resp = acct_get(f"servicePrincipals/{sp_application_id}/credentials/secrets")
        if secrets_resp.status_code != 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not list OAuth secrets for {sp_application_id!r}: "
                    f"{secrets_resp.status_code} {secrets_resp.text[:200]}."
                ),
            )
        secrets = secrets_resp.json().get("secrets") or []
        if not secrets:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "The SP has no OAuth M2M secrets. Create one via POST "
                    f"/api/2.0/accounts/{{account_id}}/servicePrincipals/"
                    f"{sp_application_id}/credentials/secrets."
                ),
            )

        if not workspace_id:
            return CheckpointResult(
                status="fail",
                error_reason="workspace_id was not resolved in setup. Re-run the setup cell.",
            )
        assign_resp = acct_get(f"workspaces/{workspace_id}/permissionassignments")
        if assign_resp.status_code != 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not list workspace permission assignments: "
                    f"{assign_resp.status_code} {assign_resp.text[:200]}."
                ),
            )
        found_assignment = False
        for pa in (assign_resp.json().get("permission_assignments") or []):
            principal = pa.get("principal") or {}
            pid = str(principal.get("service_principal_id") or principal.get("application_id") or "")
            if pid == str(sp_application_id):
                found_assignment = True
                break
        if not found_assignment:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"SP {sp_application_id!r} is not assigned to workspace "
                    f"{workspace_id}. PUT /workspaces/{{wid}}/permissionassignments/"
                    f"principals/{{app_id}} with permissions=[USER]."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: the SP row, the OAuth secret, or the workspace assignment. Each is a separate account-API call.

</details>

<details><summary>Hint 2 (direction)</summary>

Three calls in sequence: `scim_post("ServicePrincipals", ...)` returns the SP with an `applicationId`; `acct_post(f"servicePrincipals/{app_id}/credentials/secrets", {})` returns the one-time secret; `acct_put(f"workspaces/{wid}/permissionassignments/principals/{app_id}", {"permissions": ["USER"]})` assigns the SP to the workspace.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global sp_internal_id, sp_application_id

sp_resp = scim_post("ServicePrincipals", {
    "schemas": ["urn:ietf:params:scim:schemas:core:2.0:ServicePrincipal"],
    "displayName": SP_DISPLAY_NAME,
    "active": True,
})
if sp_resp.status_code == 201:
    sp_payload = sp_resp.json()
else:
    sp_payload = scim_get(
        "ServicePrincipals",
        params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'},
    ).json()["Resources"][0]
sp_internal_id = sp_payload["id"]
sp_application_id = sp_payload.get("applicationId")

secret_resp = acct_post(
    f"servicePrincipals/{sp_application_id}/credentials/secrets", {}
)
# DO NOT print secret_resp.json()["secret"]. Capture it via getpass-style
# into your local CLI profile in Task 2.

acct_put(
    f"workspaces/{workspace_id}/permissionassignments/principals/{sp_application_id}",
    {"permissions": ["USER"]},
)
```

</details>

**Wallet check.** Account API calls are free. Cumulative session damage: still under five cents (a few statement-execution calls earlier).

## Task 2: Configure the local CLI profile and deploy the Lab 8 bundle as the SP

This step happens in your LOCAL shell, not in this notebook. The notebook validates the outcome (the deployed job's `creator_user_name` and the bundle artifacts path under the SP's workspace home).

**Step A.** Append a `[labrun-sp]` block to your local `~/.databrickscfg`:

```
[labrun-sp]
host = https://YOUR-WORKSPACE-HOST
client_id = <sp_application_id from Task 1>
client_secret = <secret captured in Task 1>
auth_type = oauth-m2m
```

**Step B.** Verify the profile authenticates as the SP:

```bash
databricks current-user me --profile labrun-sp
```

This must return a Me object whose `displayName` is `labrun-cicd-deployer`.

**Step C.** Deploy the Lab 8 bundle to the prod target as the SP. From inside your `labrun-bundle/` directory:

```bash
databricks bundle validate --target prod --profile labrun-sp
databricks bundle deploy --target prod --profile labrun-sp
```

The deploy writes the bundle artifacts to `/Workspace/Users/<sp-application-id>/.bundle/labrun-bundle/prod/` (the SP's workspace home; not your user home). Capture the deployed job's `job_id` from `w.jobs.list()` (the prod job is named `labrun-bundle-job` with no `[dev <user>]` prefix). Assign it to `prod_job_id` so Task 3 can trigger a run.

The SP must also have `CAN MANAGE` on the job for the run to attribute to it. The bundle's deploy contract sets `creator_user_name` to the deploying principal, so just deploying as the SP is enough; the lab does NOT need an extra GRANT step.

In [ ]:
# NBVAL_SKIP
# Task 2: After deploying from your local CLI, capture the deployed
# prod-target job_id and confirm the artifacts path exists under the SP's
# home.

global prod_job_id

# YOUR CODE: List workspace jobs and find the one named exactly
# "labrun-bundle-job" (the bundle's prod-mode name; no [dev <user>] prefix).
# Assign its job_id to prod_job_id.
# Example:
#   for job in w.jobs.list():
#       name = (job.settings.name or "") if job.settings else ""
#       if name == JOB_NAME_HINT:
#           prod_job_id = job.job_id
#           break

if prod_job_id is None:
    print("prod_job_id is None.")
    print("Run `databricks bundle deploy --target prod --profile labrun-sp`")
    print("from your local shell with cwd = labrun-bundle/, then re-run this cell.")
else:
    info = w.jobs.get(prod_job_id)
    creator = info.creator_user_name or ""
    print(f"Prod job found: id={prod_job_id} name={info.settings.name!r}")
    print(f"Creator (must be SP applicationId, not your SSO email): {creator}")
    print(f"Tags: {info.settings.tags}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: SP profile authenticates as the SP. Prod job exists; its
# creator_user_name is the SP's applicationId (or displayName) and NOT
# the student's SSO email.


def checkpoint_2(session):
    try:
        if prod_job_id is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "prod_job_id is None. Run `databricks bundle deploy --target prod "
                    "--profile labrun-sp` from your local shell."
                ),
            )
        info = w.jobs.get(prod_job_id)
        if not info or not info.settings:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not fetch job {prod_job_id}.",
            )
        name = info.settings.name or ""
        if JOB_NAME_HINT not in name:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job name {name!r} does not contain {JOB_NAME_HINT!r}. "
                    f"Did you capture the wrong job?"
                ),
            )
        if "[dev " in name:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job name {name!r} carries the bundle-dev '[dev <user>]' prefix. "
                    f"You captured the dev job, not prod."
                ),
            )
        creator = (info.creator_user_name or "").strip()
        if not creator:
            return CheckpointResult(
                status="fail",
                error_reason=f"Job {prod_job_id} has no creator_user_name.",
            )
        # The creator should be the SP's identifier, NOT the student's SSO email.
        if creator.lower() == CALLER_USER.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job creator_user_name is {creator!r}, your SSO email. The "
                    f"deploy must run as the SP profile. Re-deploy with "
                    f"--profile labrun-sp."
                ),
            )
        # The creator should reference the SP's application_id or display name.
        if (
            sp_application_id and sp_application_id not in creator
            and SP_DISPLAY_NAME not in creator
        ):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job creator_user_name is {creator!r}; expected the SP's "
                    f"applicationId {sp_application_id!r} or displayName "
                    f"{SP_DISPLAY_NAME!r}. Confirm --profile labrun-sp was used."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: prod_job_id, the job's name shape, or a creator that matches your SSO email instead of the SP. Each is a separate fix.

</details>

<details><summary>Hint 2 (direction)</summary>

If the job's `creator_user_name` matches your email, the deploy ran as your user PAT, not the SP. Check `~/.databrickscfg` , the `[labrun-sp]` block must have `auth_type = oauth-m2m` AND your shell must pass `--profile labrun-sp` to the deploy command. Re-deploy and re-fetch.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global prod_job_id
for job in w.jobs.list():
    name = (job.settings.name or "") if job.settings else ""
    if name == JOB_NAME_HINT:
        prod_job_id = job.job_id
        break
```

If the job is missing entirely, the bundle deploy did not run; the local shell will show the error. Common cause: the `[labrun-sp]` block has the wrong host or a typo in the client_id/secret. Run `databricks current-user me --profile labrun-sp` to validate the profile first.

</details>

**Wallet check.** Still under $0.10 cumulative. The bundle deploy is metadata-only; the workspace warehouse spent a few seconds running the validation.

## Task 3: Trigger one run of the prod job as the SP and wait for SUCCESS

In your local shell:

```bash
databricks jobs run-now --job-id <prod_job_id> --profile labrun-sp
```

The output includes a `run_id`. Paste it into `run_id` below so the notebook can poll the run state and confirm `state.life_cycle_state=TERMINATED` and `state.result_state=SUCCESS`. The Lab 8 bundle job runs in well under 90 seconds, so the polling loop has a generous ceiling.

The run's `creator_user_name` must also be the SP. The notebook validates this from the workspace Runs API (`GET /api/2.1/jobs/runs/get?run_id=<id>`).

In [ ]:
# NBVAL_SKIP
# Task 3: Trigger the run via CLI on your local shell, paste the run_id
# here, wait for SUCCESS.

global run_id

# YOUR CODE: Assign run_id to the integer run_id returned by
# `databricks jobs run-now --job-id <prod_job_id> --profile labrun-sp`.
# run_id = 1234567890

if run_id is None:
    print("run_id is None. Trigger the run via CLI and paste the run_id above.")
    raise SystemExit(1)

# Poll the run until it terminates. Ceiling 6 minutes.
deadline = time.time() + 360
last_state = None
while time.time() < deadline:
    info = w.jobs.get_run(run_id)
    state = info.state
    life = state.life_cycle_state.value if (state and state.life_cycle_state) else None
    result = state.result_state.value if (state and state.result_state) else None
    if life != last_state:
        print(f"  life_cycle_state={life}  result_state={result}")
        last_state = life
    if life in ("TERMINATED", "INTERNAL_ERROR", "SKIPPED"):
        break
    time.sleep(5)

info = w.jobs.get_run(run_id)
final_state = info.state
print()
print(f"Run {run_id} final: life={final_state.life_cycle_state.value if final_state.life_cycle_state else None} "
      f"result={final_state.result_state.value if final_state.result_state else None}")
print(f"Run creator: {info.creator_user_name!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Run reached TERMINATED + SUCCESS, and creator_user_name
# is the SP (not the student's SSO email).


def checkpoint_3(session):
    try:
        if run_id is None:
            return CheckpointResult(
                status="fail",
                error_reason="run_id is None. Trigger the run via CLI and capture its id.",
            )
        info = w.jobs.get_run(run_id)
        state = info.state
        if not state or not state.life_cycle_state:
            return CheckpointResult(
                status="fail",
                error_reason=f"Run {run_id} has no life_cycle_state.",
            )
        if state.life_cycle_state.value != "TERMINATED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run {run_id} life_cycle_state is "
                    f"{state.life_cycle_state.value!r}; expected TERMINATED. "
                    f"The polling loop should have waited; re-run this cell."
                ),
            )
        result_state = state.result_state.value if state.result_state else None
        if result_state != "SUCCESS":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run {run_id} result_state is {result_state!r}; expected SUCCESS. "
                    f"Open the run in the workspace UI to see the failure detail."
                ),
            )
        creator = (info.creator_user_name or "").strip()
        if creator.lower() == CALLER_USER.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run creator_user_name is {creator!r}, your SSO email. "
                    f"The run must be triggered as the SP. Re-trigger with "
                    f"--profile labrun-sp."
                ),
            )
        if (
            sp_application_id and sp_application_id not in creator
            and SP_DISPLAY_NAME not in creator
        ):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run creator_user_name is {creator!r}; expected the SP's "
                    f"applicationId {sp_application_id!r} or displayName "
                    f"{SP_DISPLAY_NAME!r}."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is wrong: run_id missing, run not terminated, result not SUCCESS, or creator is your email instead of the SP. Each is a separate diagnosis.

</details>

<details><summary>Hint 2 (direction)</summary>

If `result_state` is FAILED, open the run in the workspace UI (`Workflows -> Runs -> <run_id>`) and look at the task logs; the Lab 8 bundle's notebook is small but ANY missing UC permission can fail the task. If the creator is your SSO email, you triggered the run with the wrong CLI profile. Re-run `databricks jobs run-now --job-id <id> --profile labrun-sp`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global run_id
# Paste the integer printed by `databricks jobs run-now ... --profile labrun-sp`.
run_id = 1234567890
```

Then re-run the polling cell; the loop blocks until life_cycle_state ends.

</details>

**Wallet check.** Still under $0.30 cumulative. One job run on serverless compute against the Premium trial workspace; a few cents at most.

## Task 4: Query the audit log and prove the SP is the actor on both deploy and run

The audit log lives at `system.access.audit` (the Unity Catalog system table for audit). The `bundle.deploy` and `jobs.runNow` events surface with `action_name` plus a `user_identity` JSON object containing `email` (for human users) or `subject` (for service principals).

You need SELECT on `system.access`. If your account does not already have it, grant it via:

```sql
GRANT SELECT ON SCHEMA system.access TO `<your-account-admin-or-metastore-admin-user>`;
```

(You may need account admin or metastore admin to GRANT. Confirm with your trial setup.)

Then run two queries:

1. Filter the last 60 minutes for `action_name IN ('jobs.runNow', 'bundle.deploy', 'workspace.workspaceConfDeploy')` and assert every row's `user_identity` references the SP (its `applicationId` or `displayName`).
2. Run a control query for the student's SSO email on the same action_names in the last 60 minutes; the result must be ZERO rows (proving you did not also deploy or run from your personal profile during this session).

The audit log has 1 to 5 minutes of lag. The checkpoint retries for 6 minutes with quirky wait messages.

In [ ]:
# NBVAL_SKIP
# Task 4: Two SELECTs against system.access.audit. Print results; the
# checkpoint re-runs the same queries with retries to absorb log lag.

# YOUR CODE: SELECT action_name, user_identity FROM system.access.audit
# WHERE action_name IN ('jobs.runNow','bundle.deploy','workspace.workspaceConfDeploy')
# AND event_time > current_timestamp() - INTERVAL 60 MINUTES
# AND workspace_id = <workspace_id>
# Inspect rows; confirm user_identity references SP_DISPLAY_NAME or sp_application_id.

# Helper for the student to run while reading the log:
sp_query = (
    "SELECT action_name, user_identity "
    "FROM system.access.audit "
    "WHERE action_name IN ('jobs.runNow', 'bundle.deploy', 'workspace.workspaceConfDeploy') "
    "AND event_time > current_timestamp() - INTERVAL 60 MINUTES "
    f"AND workspace_id = {workspace_id}"
)

print("Asking the audit log to catch up...")
for attempt in range(6):
    res = run_sql(sp_query, ignore_errors=True)
    if res.get("error"):
        print(f"  audit query failed: {res['error']}")
        break
    rows = res.get("rows") or []
    if rows:
        print(f"  found {len(rows)} matching event(s):")
        for row in rows:
            action, identity = row
            print(f"    {action}: {identity}")
        break
    print(f"  attempt {attempt+1}: no matching events yet, waiting 30s...")
    time.sleep(30)
else:
    print("  audit log did not surface events in 3 minutes; checkpoint will retry.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: At least 2 audit events for the SP on the relevant action
# names; zero audit events for the student's SSO email on those names.


def checkpoint_4(session):
    try:
        if not workspace_id:
            return CheckpointResult(
                status="fail",
                error_reason="workspace_id was not resolved in setup.",
            )

        identifiers_for_sp = [
            v for v in (SP_DISPLAY_NAME, sp_application_id) if v
        ]

        sp_filter_terms = " OR ".join(
            f"user_identity:email = '{ident}' OR user_identity:subject = '{ident}'"
            for ident in identifiers_for_sp
        )

        action_filter = (
            "action_name IN ('jobs.runNow', 'bundle.deploy', "
            "'workspace.workspaceConfDeploy')"
        )

        # Retry loop for audit log lag.
        deadline = time.time() + 360
        sp_rows = []
        student_rows = []
        while time.time() < deadline:
            sp_sql = (
                "SELECT action_name FROM system.access.audit "
                f"WHERE {action_filter} "
                "AND event_time > current_timestamp() - INTERVAL 60 MINUTES "
                f"AND workspace_id = {workspace_id} "
                f"AND ({sp_filter_terms})"
            )
            sp_res = run_sql(sp_sql, ignore_errors=True)
            if sp_res.get("error") and "INSUFFICIENT_PERMISSIONS" in (sp_res.get("error") or ""):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        "SELECT on system.access.audit is denied. Run "
                        "GRANT SELECT ON SCHEMA system.access TO `<your-user>` "
                        "as metastore admin and retry."
                    ),
                )
            if sp_res.get("error"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"audit query failed: {sp_res['error']}",
                )
            sp_rows = sp_res.get("rows") or []

            student_sql = (
                "SELECT action_name FROM system.access.audit "
                f"WHERE {action_filter} "
                "AND event_time > current_timestamp() - INTERVAL 60 MINUTES "
                f"AND workspace_id = {workspace_id} "
                f"AND user_identity:email = '{CALLER_USER}'"
            )
            student_res = run_sql(student_sql, ignore_errors=True)
            student_rows = student_res.get("rows") or []

            if len(sp_rows) >= 2:
                break
            time.sleep(30)

        if len(sp_rows) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"After 6 minutes of polling, only {len(sp_rows)} audit row(s) "
                    f"attribute action_name in (jobs.runNow, bundle.deploy, "
                    f"workspace.workspaceConfDeploy) to the SP. Need at least 2 "
                    f"(one deploy + one runNow). Confirm both Tasks 2 and 3 ran as "
                    f"--profile labrun-sp."
                ),
            )

        sp_actions = {row[0] for row in sp_rows}
        if "jobs.runNow" not in sp_actions:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Audit shows no jobs.runNow event attributed to the SP. "
                    "Re-trigger the run with --profile labrun-sp."
                ),
            )
        if not ({"bundle.deploy", "workspace.workspaceConfDeploy"} & sp_actions):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Audit shows no bundle.deploy or workspace.workspaceConfDeploy "
                    "event attributed to the SP. Re-deploy with --profile labrun-sp."
                ),
            )

        if student_rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Audit shows {len(student_rows)} matching event(s) attributed "
                    f"to your SSO email {CALLER_USER!r}. This is a parallel deploy "
                    f"or run from your personal profile during the lab. Clean up "
                    f"any extra deploys and re-check."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: a permission grant on `system.access`, audit rows for the SP, or rows that wrongly attribute to your email. Each maps to a separate fix.

</details>

<details><summary>Hint 2 (direction)</summary>

If you get `INSUFFICIENT_PERMISSIONS`, run `GRANT SELECT ON SCHEMA system.access TO ` followed by your account email in backticks. Audit row attribution depends on `user_identity` JSON; for SPs the relevant field is `subject` (the application_id) plus `email` (the displayName on some runtimes). The lab's filter accepts either.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```sql
GRANT SELECT ON SCHEMA system.access TO `your-account-admin-email@example.com`;
```

Then re-run the checkpoint. The 6-minute retry loop absorbs the audit lag.

If you accidentally deployed from your personal profile during the session, run:

```bash
databricks bundle destroy --target prod --profile <your-personal-profile> --auto-approve
```

Then re-deploy with `--profile labrun-sp` so the only matching audit rows attribute to the SP.

</details>

**Wallet check.** Still under $0.50 cumulative. The audit queries scan a small slice of `system.access.audit`; on a fresh trial workspace the scan size is tiny.

## Cleanup

The cleanup cell tears down (in critical-first order):

1. The deployed prod-target job (running deploys consume daily quota).
2. The SP's workspace bundle artifacts (`/Workspace/Users/<sp-application-id>/.bundle/...`).
3. The SP's OAuth M2M secret.
4. The SP's workspace assignment.
5. The SP itself (account-scoped).

Manual reminders the cleanup cell prints:

- Remove the `[labrun-sp]` block from your local `~/.databrickscfg` (the cleanup cell does not edit local files).
- Remove any cloud-side IAM role / managed identity / service account created back in Lab 10.
- Tear down the Premium trial workspace before day 14 of the trial. This is the LAST Premium-trial lab in the cert track.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down everything in CLEANUP_MANIFEST.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")
print()
print("MANUAL CLEANUP REMINDERS:")
print("  1. Remove the [labrun-sp] block from your local ~/.databrickscfg")
print("     (the cleanup cell does not edit local files).")
print("  2. Remove any cloud-side IAM role / managed identity / service account")
print("     created in Lab 10.")
print("  3. Tear down the Premium trial workspace before day 14 of the trial.")
print("     This is the last Premium-trial lab in the cert track.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.10 to $0.50 on your cloud account.** The Premium trial itself stayed at $0; the warehouse, the bundle deploy, and the one job run added up to under fifty cents. The bigger line item, if you skip the workspace teardown before day 14, is the trial converting to paid for the rest of the month at roughly $5 to $30 per day.

## Reflection

These are not graded. They are for you.

1. Walk through what happens when `databricks jobs run-now --job-id <id> --profile labrun-sp` runs. Name each step: the CLI looks up the profile in `~/.databrickscfg`, fetches an OAuth M2M token, calls the Jobs API with the token, the API authenticates the token and resolves the principal, the run is created and attributed. At which step does the audit log get the principal's identity, and why is that not the same as your SSO email?

2. Your team currently has six CI/CD pipelines running as the team lead's PAT. Sketch the migration to service principal auth: how many principals do you create, what is granted to each, and how do you migrate the existing pipelines without breaking any running deploys? Why is "one principal per pipeline" a reasonable default vs "one shared principal for all CI/CD"?

## Exam-Style Practice

**Question 1 (Domain 5 / 7, CI/CD auth principal):**

A team wants their CI/CD pipeline to deploy Declarative Automation Bundles to prod. The audit log must show a service identity as the actor on every deploy, the credential must rotate without redoing the pipeline config, and a human's PAT must NOT be involved. Which Databricks auth mechanism fits?

A. A personal access token (PAT) created by the team's automation user; rotate monthly.
B. OAuth M2M (machine-to-machine) authentication using a service principal's client ID and client secret stored as CI/CD environment variables.
C. SSO with the team lead's identity, automated via a saved browser session in the CI/CD runner.
D. A workspace admin's `~/.databrickscfg` file copied into the CI/CD runner.

<details><summary>Show answer</summary>

**Correct: B.**

- A is the anti-pattern the scenario explicitly rules out. PAT auth attributes deploys to the PAT's owner; even an "automation user" is a human-shaped account.
- B is correct: OAuth M2M with a service principal is the documented Databricks CI/CD authentication pattern. The principal is a non-human identity, the audit log attributes deploys to the principal, the secret rotates via the account API without changing the pipeline config beyond the secret store, and no human PAT is involved.
- C is wrong: SSO requires a browser session that CI/CD does not have. Saving a session into a runner is fragile and is also an audit anti-pattern.
- D is wrong: copying a human's `~/.databrickscfg` into a CI/CD runner is functionally a stolen credential. The audit log would attribute deploys to the human.

</details>

**Question 2 (Domain 7, audit log attribution):**

A platform engineer queries `system.access.audit` for `action_name='jobs.runNow'` in the last hour and sees rows whose `user_identity.email` is `labrun-cicd-deployer` (a service principal's identifier). The engineer wants to confirm that the SOC 2 auditor will see the principal as the actor and not a "Databricks system" generic identity. What is the auditor most likely to see?

A. Every jobs.runNow event has the service principal as the user_identity; the auditor sees a clear actor trail.
B. The user_identity is the human who originally created the service principal; Databricks resolves up to the creator.
C. The user_identity is "Databricks system" for any non-human actor; SOC 2 auditors typically accept this as "automation."
D. The audit log does not capture jobs.runNow events; only data-access events surface in `system.access.audit`.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: the audit log attributes the actor to whichever principal authenticated the API call. A service principal making the call gets attributed to the principal, period. SOC 2 auditors get a clear actor trail through the principal's display name and application_id.
- B is wrong: the audit log does NOT resolve the actor to the human who created the principal. The principal IS the actor.
- C is wrong: there is no generic "Databricks system" identity for principal-authenticated events. System events have a separate `service_name` field, but actor-driven events name the actor.
- D is wrong: `system.access.audit` captures jobs.runNow and most workspace-level events; it is not limited to data-access events.

</details>